# Assignment 4
Naila Salma Yusroini

24/536726/PA/22766

In [ ]:
!pip install yt-dlp imageio[ffmpeg]
!yt-dlp -f "bv*[ext=mp4]+ba[ext=m4a]/b[ext=mp4]" --merge-output-format mp4 "https://www.youtube.com/watch?v=ORrrKXGx2SE" -o "traffic_data.mp4"

# Interactive Coordinate Selection
Before putting it in to use template matching and optical flow, first a bounding box is implemented to one of the frames in the video to select an object that wants to be detected.

In [ ]:
import cv2
import numpy as np
import imageio
import IPython
from google.colab import output
import base64

video_path = 'traffic_data.mp4'

reader = imageio.get_reader(video_path, format='ffmpeg')
first_frame_rgb = reader.get_data(0)
reader.close()

max_width = 800
h, w, _ = first_frame_rgb.shape
if w > max_width:
    scale = max_width / w
    first_frame_rgb = cv2.resize(first_frame_rgb, (int(w * scale), int(h * scale)))
    h, w, _ = first_frame_rgb.shape

_, encoded_img = cv2.imencode('.jpg', cv2.cvtColor(first_frame_rgb, cv2.COLOR_RGB2BGR))
img_base64 = base64.b64encode(encoded_img).decode('utf-8')

js_code = f"""
<div style="font-family: Arial, sans-serif; margin-bottom: 10px;">
    <strong>Interactive ROI Chooser (Colab Safe)</strong><br>
    Instructions: Click the <strong>Top-Left</strong> corner, then click the <strong>Bottom-Right</strong> corner of your target object.
</div>
<canvas id="canvas" width="{w}" height="{h}" style="border:2px solid #00ff00; cursor: crosshair;"></canvas>

<script>
    var canvas = document.getElementById('canvas');
    var ctx = canvas.getContext('2d');
    var img = new Image();

    img.onload = function() {{
        ctx.drawImage(img, 0, 0);
    }};
    img.src = 'data:image/jpeg;base64,{img_base64}';

    var clicks = [];
    canvas.onclick = function(e) {{
        var rect = canvas.getBoundingClientRect();
        var x = e.clientX - rect.left;
        var y = e.clientY - rect.top;
        clicks.push({{x: x, y: y}});

        ctx.beginPath();
        ctx.arc(x, y, 4, 0, 2 * Math.PI);
        ctx.fillStyle = "#ff0000";
        ctx.fill();

        if (clicks.length === 2) {{
            var x_min = Math.min(clicks[0].x, clicks[1].x);
            var y_min = Math.min(clicks[0].y, clicks[1].y);
            var x_max = Math.max(clicks[0].x, clicks[1].x);
            var y_max = Math.max(clicks[0].y, clicks[1].y);
            var width = x_max - x_min;
            var height = y_max - y_min;

            ctx.strokeStyle = "#00ff00";
            ctx.lineWidth = 2;
            ctx.strokeRect(x_min, y_min, width, height);

            google.colab.kernel.invokeFunction('notebook.save_coords', [x_min, y_min, width, height], {{}});
        }}
    }};
</script>
"""

selected_x, selected_y, selected_w, selected_h = 0, 0, 0, 0

def save_coords(x, y, width, height):
    global selected_x, selected_y, selected_w, selected_h
    selected_x, selected_y = int(x), int(y)
    selected_w, selected_h = int(width), int(height)
    print(f"\n Coordinates Saved: x={selected_x}, y={selected_y}, w={selected_w}, h={selected_h}")

output.register_callback('notebook.save_coords', save_coords)
IPython.display.display(IPython.display.HTML(js_code))

# Template Matching

In [ ]:
import cv2
import numpy as np
import imageio
from google.colab.patches import cv2_imshow

video_path = 'traffic_data.mp4'

try:
    reader = imageio.get_reader(video_path, format='ffmpeg')

    first_frame = reader.get_data(0)
    first_frame_bgr = cv2.cvtColor(first_frame, cv2.COLOR_RGB2BGR)
    gray_first = cv2.cvtColor(first_frame_bgr, cv2.COLOR_BGR2GRAY)

    orig_h, orig_w = gray_first.shape
    canvas_w = 800 if orig_w > 800 else orig_w
    scale_factor = orig_w / canvas_w

    x = int(selected_x * scale_factor)
    y = int(selected_y * scale_factor)
    w = int(selected_w * scale_factor)
    h = int(selected_h * scale_factor)

    print(f"Original Video Resolution: {orig_w}x{orig_h}")
    print(f"Scaled Coordinates: x={x}, y={y}, w={w}, h={h}")

    template = gray_first[y:y+h, x:x+w]

    print("This is the template the algorithm is tracking:")
    cv2_imshow(template)
    print("\n Running Template Matching Tracking...")

    for frame_count, frame_rgb in enumerate(reader):
        if frame_count >= 30:
            break

        frame_bgr = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)
        gray_frame = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)

        result = cv2.matchTemplate(gray_frame, template, cv2.TM_CCOEFF_NORMED)
        _, _, _, max_loc = cv2.minMaxLoc(result)

        top_left = max_loc
        bottom_right = (top_left[0] + w, top_left[1] + h)

        cv2.rectangle(frame_bgr, top_left, bottom_right, (0, 255, 0), 2)

        print(f"Template Frame {frame_count}")
        cv2_imshow(frame_bgr)

    reader.close()

except Exception as e:
    print(f"Error: {e}")

### Analysis
After the bounding box is implemented, the person that we want to detect is now much more clearer to detect. With template matching, it's able to identify and detect the person moving with that same bounding box that was first implemented

# Optical Flow

In [ ]:
import cv2
import numpy as np
import imageio
from google.colab.patches import cv2_imshow

video_path = 'traffic_data.mp4'

try:
    reader = imageio.get_reader(video_path, format='ffmpeg')

    first_frame = reader.get_data(0)
    old_frame = cv2.cvtColor(first_frame, cv2.COLOR_RGB2BGR)
    old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)

    orig_h, orig_w = old_gray.shape
    canvas_w = 800 if orig_w > 800 else orig_w
    scale_factor = orig_w / canvas_w

    x = int(selected_x * scale_factor)
    y = int(selected_y * scale_factor)
    w = int(selected_w * scale_factor)
    h = int(selected_h * scale_factor)

    print(f"Original Video Resolution: {orig_w}x{orig_h}")
    print(f"Tracking Region Set: x={x}, y={y}, w={w}, h={h}")

    # Lucas-Kanade parameters
    feature_params = dict(maxCorners=30, qualityLevel=0.3, minDistance=5, blockSize=5)
    lk_params = dict(winSize=(15, 15), maxLevel=2,
                     criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))

    mask = np.zeros_like(old_gray)
    mask[y:y+h, x:x+w] = 255

    p0 = cv2.goodFeaturesToTrack(old_gray, mask=mask, **feature_params)
    track_mask = np.zeros_like(old_frame)

    if p0 is None:
        print("Warning: No trackable corner features found inside your selected box.")
        print("The region might be too smooth. Try drawing a slightly wider box to capture more clothing/edge contrast!")
    else:
        print(f"Found {len(p0)} keypoints to track inside the target area.")
        print("Running Optical Flow Tracking...")

    for frame_count, frame_rgb in enumerate(reader):
        if frame_count == 0:
            continue
        if frame_count >= 30:
            break

        frame = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)
        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        p1, st, err = cv2.calcOpticalFlowPyrLK(old_gray, frame_gray, p0, None, **lk_params)

        if p1 is not None and len(p1) > 0:
            good_new = p1[st == 1]
            good_old = p0[st == 1]

            for i, (new, old) in enumerate(zip(good_new, good_old)):
                a, b = new.ravel().astype(int)
                c, d = old.ravel().astype(int)
                track_mask = cv2.line(track_mask, (a, b), (c, d), (0, 255, 255), 2)
                frame = cv2.circle(frame, (a, b), 4, (0, 0, 255), -1)

            img = cv2.add(frame, track_mask)
            print(f"Optical Flow Frame {frame_count}")
            cv2_imshow(img)

            old_gray = frame_gray.copy()
            p0 = good_new.reshape(-1, 1, 2)
        else:
            print(f"Tracking completely lost on frame {frame_count}")
            cv2_imshow(frame)

    reader.close()
    print("Optical Flow analysis run complete!")

except Exception as e:
    print(f"Error: {e}")

### Analysis
After running the Lucas-Kanade Optical Flow the detector isolated exactly 6 feature points corresponding to distinct contrast textures on the person's clothing and shoulder. By tracking 6 points instead of a single point gives an advantage of being able to tell the difference in a person's movements.